# 🎓 MentorX — Llama-3.1-8B-Instruct SFT (QLoRA)

**Model:** unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit  
**Yöntem:** QLoRA (4-bit) via Unsloth  
**Veri:** 1824 Türkçe Sokratik diyalog (Otomata Teorisi)  
**Hedef:** MentorX Sokratik tutor davranışını öğretmek  
**Şartlar:** Qwen2.5-7B Automata ile birebir aynı

> ⚠️ **Colab Pro / A100 (40GB) önerilir.**

## 📦 1. Kurulum

In [1]:
%%capture
!pip install -U unsloth wandb

In [2]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## 🔧 2. Konfigürasyon

In [3]:
# Model
MODEL_NAME       = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH   = 2048
DTYPE            = None
LOAD_IN_4BIT     = True

# LoRA — Qwen2.5-7B Automata ile aynı
LORA_R           = 16
LORA_ALPHA       = 16
LORA_DROPOUT     = 0.0
TARGET_MODULES   = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"]

# Eğitim — Qwen2.5-7B Automata ile aynı
OUTPUT_DIR       = "./mentorx-llama3.1-8b-lora"
NUM_EPOCHS       = 3
BATCH_SIZE       = 4
GRAD_ACCUM       = 4
LEARNING_RATE    = 2e-4
WARMUP_STEPS     = 50
LR_SCHEDULER     = "cosine"
WEIGHT_DECAY     = 0.01
MAX_GRAD_NORM    = 1.0
LOGGING_STEPS    = 10
SEED             = 42

# Veri
TRAIN_FILE       = "train_v2.jsonl"
VAL_FILE         = "val_v2.jsonl"

# HuggingFace Hub
HF_PUSH          = True
HF_REPO          = "mtepe01/mentorx-llama3.1-8b-automata"

# WandB
USE_WANDB        = False
WANDB_PROJECT    = "mentorx-sft"

print("✅ Konfigürasyon yüklendi")
print(f"   Model        : {MODEL_NAME}")
print(f"   LORA_R       : {LORA_R} | LORA_ALPHA: {LORA_ALPHA}")
print(f"   Effective BS : {BATCH_SIZE * GRAD_ACCUM}")

✅ Konfigürasyon yüklendi
   Model        : unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
   LORA_R       : 16 | LORA_ALPHA: 16
   Effective BS : 16


## 📂 3. Veri Yükleme

In [4]:
from google.colab import files

print("train_v2.jsonl ve val_v2.jsonl dosyalarını seç:")
uploaded = files.upload()

for fname in uploaded:
    print(f"✅ Yüklendi: {fname} ({len(uploaded[fname])/1024:.1f} KB)")

train_v2.jsonl ve val_v2.jsonl dosyalarını seç:


Saving train_v2.jsonl to train_v2.jsonl
Saving val_v2.jsonl to val_v2.jsonl
✅ Yüklendi: train_v2.jsonl (8091.7 KB)
✅ Yüklendi: val_v2.jsonl (886.2 KB)


In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE}
)

print(f"Train örnekleri : {len(dataset['train'])}")
print(f"Val örnekleri   : {len(dataset['validation'])}")

sample = dataset["train"][0]
print(f"\n--- Örnek diyalog ({len(sample['messages'])} mesaj) ---")
for msg in sample["messages"][:3]:
    role = msg["role"].upper()
    preview = msg["content"][:80] + "..." if len(msg["content"]) > 80 else msg["content"]
    print(f"[{role}]: {preview}")

def validate_split(ds, name):
    for i, row in enumerate(ds):
        msgs = row.get("messages")
        assert isinstance(msgs, list) and len(msgs) >= 3, f"{name}[{i}] invalid messages"
        assert msgs[0].get("role") == "system",    f"{name}[{i}] first role must be system"
        assert msgs[-1].get("role") == "assistant", f"{name}[{i}] last role must be assistant"
        for j, m in enumerate(msgs):
            assert m.get("role") in {"system", "user", "assistant"}, f"{name}[{i}] bad role @{j}"
            c = m.get("content")
            assert isinstance(c, str) and c.strip(), f"{name}[{i}] empty content @{j}"

validate_split(dataset["train"],      "train")
validate_split(dataset["validation"], "validation")
print("✅ Dataset şeması doğrulandı")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Train örnekleri : 1824
Val örnekleri   : 201

--- Örnek diyalog (13 mesaj) ---
[SYSTEM]: Sen MentorX adlı bir Türkçe Otomata Teorisi tutorusun. Öğrenciye asla doğrudan c...
[USER]: Hocam, `b ∪ Σ*a` dilini tanıyan bir NFA taslağı düşündüm. Başlangıçtan iki kola ...
[ASSISTANT]: Güzel bir başlangıç taslağı. Peki, bu taslağınızın 'ab' dizisini kabul etmemesi ...
✅ Dataset şeması doğrulandı


## 🤖 4. Model & Tokenizer Yükleme

In [6]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = DTYPE,
    load_in_4bit  = LOAD_IN_4BIT,
)

print(f"✅ Model yüklendi: {MODEL_NAME}")
print(f"   Parametre sayısı: {model.num_parameters()/1e9:.2f}B")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


✅ Model yüklendi: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
   Parametre sayısı: 8.03B


## 🔗 5. LoRA Adapter Ekleme

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = TARGET_MODULES,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
    use_rslora                 = False,
    loftq_config               = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA eklendi")
print(f"   Eğitilebilir: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)")

Unsloth 2026.6.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ LoRA eklendi
   Eğitilebilir: 41.9M / 4.58B (0.92%)


## 🗂️ 6. Veri Formatı — Chat Template

In [8]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def format_conversations(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize              = False,
            add_generation_prompt = False,
        )
        texts.append(text)
    return {"text": texts}

dataset_formatted = dataset.map(
    format_conversations,
    batched        = True,
    remove_columns = dataset["train"].column_names,
)

print("✅ Chat template uygulandı")
print("\n--- Formatlanmış örnek (ilk 500 karakter) ---")
print(dataset_formatted["train"][0]["text"][:500])
print("...")

Map:   0%|          | 0/1824 [00:00<?, ? examples/s]

Map:   0%|          | 0/201 [00:00<?, ? examples/s]

✅ Chat template uygulandı

--- Formatlanmış örnek (ilk 500 karakter) ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Sen MentorX adlı bir Türkçe Otomata Teorisi tutorusun. Öğrenciye asla doğrudan cevap vermezsin, her mesajında yönlendirici sorular sorarsın, max 3 cümleyle konuşursun.<|eot_id|><|start_header_id|>user<|end_header_id|>

Hocam, `b ∪ Σ*a` dilini tanıyan bir NFA taslağı düşündüm. Başlangıçtan iki kola ayrılıyorum: bir kol sadece 'b' ile nihai bir duruma gider. Diğer kol ise `Σ
...


In [9]:
import numpy as np

sample_texts = dataset_formatted["train"]["text"]
lengths = [len(tokenizer.encode(t)) for t in sample_texts]

print(f"Token uzunlukları:")
print(f"  Min    : {min(lengths)}")
print(f"  Max    : {max(lengths)}")
print(f"  Ort    : {np.mean(lengths):.0f}")
print(f"  P95    : {np.percentile(lengths, 95):.0f}")
print(f"  MAX_SEQ: {MAX_SEQ_LENGTH}")

truncated = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
print(f"  Kesilen: {truncated} / {len(lengths)} örnek ({100*truncated/len(lengths):.1f}%)")

Token uzunlukları:
  Min    : 380
  Max    : 5219
  Ort    : 1263
  P95    : 3145
  MAX_SEQ: 2048
  Kesilen: 247 / 1824 örnek (13.5%)


## 🏋️ 7. Eğitim

In [10]:
import os

if USE_WANDB:
    import wandb
    wandb.login()
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
    report_to = "wandb"
    print(f"✅ WandB aktif: {WANDB_PROJECT}")
else:
    os.environ["WANDB_DISABLED"] = "true"
    report_to = "none"
    print("WandB devre dışı")

WandB devre dışı


In [15]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq, set_seed
from unsloth import is_bfloat16_supported
import math

set_seed(SEED)

steps_per_epoch = max(1, math.ceil(len(dataset_formatted["train"]) / (BATCH_SIZE * GRAD_ACCUM)))
eval_save_steps = max(20, steps_per_epoch // 2)

data_collator = DataCollatorForSeq2Seq(
    tokenizer          = tokenizer,
    model              = model,
    padding            = "longest",
    pad_to_multiple_of = 8,
)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset_formatted["train"],
    eval_dataset       = dataset_formatted["validation"],
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,
    data_collator      = data_collator,
    args = SFTConfig(
        output_dir                   = OUTPUT_DIR,
        num_train_epochs             = NUM_EPOCHS,
        per_device_train_batch_size  = BATCH_SIZE,
        per_device_eval_batch_size   = 1,
        gradient_accumulation_steps  = GRAD_ACCUM,
        warmup_steps                 = WARMUP_STEPS,
        learning_rate                = LEARNING_RATE,
        lr_scheduler_type            = LR_SCHEDULER,
        weight_decay                 = WEIGHT_DECAY,
        max_grad_norm                = MAX_GRAD_NORM,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = LOGGING_STEPS,
        save_steps                   = eval_save_steps,
        eval_strategy                = "steps",
        eval_steps                   = eval_save_steps,
        load_best_model_at_end       = True,
        metric_for_best_model        = "eval_loss",
        greater_is_better            = False,
        save_total_limit             = 3,
        seed                         = SEED,
        report_to                    = report_to,
        optim                        = "adamw_8bit",
        dataloader_pin_memory        = True,
        dataset_text_field           = "text",
        max_seq_length               = MAX_SEQ_LENGTH,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1824 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/201 [00:00<?, ? examples/s]

In [16]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

sample   = trainer.train_dataset[0]
labels   = sample["labels"]
masked   = sum(1 for l in labels if l == -100)
unmasked = sum(1 for l in labels if l != -100)
total    = len(labels)

print(f"✅ train_on_responses_only aktif")
print(f"   Toplam token   : {total}")
print(f"   Maskelenen     : {masked}  (system + user → loss yok)")
print(f"   Loss hesaplanan: {unmasked} (sadece assistant yanıtları)")
print(f"   Oran           : %{100*unmasked/total:.1f} assistant token")

assert unmasked > 0, "Assistant mask başarısız!"
assert masked   > 0, "Mask başarısız!"
print("✅ Maskeleme doğrulandı")

Map (num_proc=16):   0%|          | 0/1824 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/1824 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/201 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/201 [00:00<?, ? examples/s]

✅ train_on_responses_only aktif
   Toplam token   : 1239
   Maskelenen     : 805  (system + user → loss yok)
   Loss hesaplanan: 434 (sadece assistant yanıtları)
   Oran           : %35.0 assistant token
✅ Maskeleme doğrulandı


In [17]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"VRAM toplam      : {max_memory:.1f} GB")
print(f"VRAM kullanım    : {start_gpu_memory:.1f} GB")

GPU: NVIDIA A100-SXM4-40GB
VRAM toplam      : 39.5 GB
VRAM kullanım    : 23.6 GB


In [18]:
s = trainer.train_dataset[0]
assert "labels" in s and any(x != -100 for x in s["labels"]), \
    "train_on_responses_only uygulanmamış — eğitimi durdur!"

print("Eğitim başlıyor...")
trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n✅ Eğitim tamamlandı!")
print(f"   Süre       : {trainer_stats.metrics['train_runtime']/60:.1f} dakika")
print(f"   Train loss : {trainer_stats.metrics['train_loss']:.4f}")
print(f"   Peak VRAM  : {used_memory:.1f} GB / {max_memory:.1f} GB")

Eğitim başlıyor...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,824 | Num Epochs = 3 | Total steps = 342
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
57,1.262413,1.270730
114,1.193066,1.215767
171,1.050530,1.197118
228,1.027084,1.163547
285,0.865823,1.188435
342,0.862236,1.184760


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


✅ Eğitim tamamlandı!
   Süre       : 38.5 dakika
   Train loss : 1.0423
   Peak VRAM  : 23.6 GB / 39.5 GB


## 💾 8. Model Kaydetme & HF Push

In [19]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ LoRA adaptörleri kaydedildi: {OUTPUT_DIR}")

Unsloth: Restored added_tokens_decoder metadata in ./mentorx-llama3.1-8b-lora/tokenizer_config.json.


✅ LoRA adaptörleri kaydedildi: ./mentorx-llama3.1-8b-lora


In [21]:
if HF_PUSH:
    from huggingface_hub import login
    login()
    model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print(f"✅ HF Hub'a yüklendi: https://huggingface.co/{HF_REPO}")
else:
    print("HF_PUSH=False — Hub'a yükleme atlandı")

README.md:   0%|          | 0.00/575 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   3%|3         | 5.59MB /  168MB            

Saved model to https://huggingface.co/mtepe01/mentorx-llama3.1-8b-automata


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpkppu4qti/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpkppu4qti/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

✅ HF Hub'a yüklendi: https://huggingface.co/mtepe01/mentorx-llama3.1-8b-automata


In [22]:
# Merge edilmiş model push et (vLLM için)
model.save_pretrained_merged(
    OUTPUT_DIR + "-merged",
    tokenizer,
    save_method = "merged_16bit",
)

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(HF_REPO + "-merged", exist_ok=True)
model.push_to_hub_merged(
    HF_REPO + "-merged",
    tokenizer,
    save_method = "merged_16bit",
)
print(f"✅ Merged model push edildi: https://huggingface.co/{HF_REPO}-merged")

config.json:   0%|          | 0.00/956 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in ./mentorx-llama3.1-8b-lora-merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:  18%|#7        |  208MB / 1.17GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [03:55<00:00, 58.91s/it]


Unsloth: Merge process complete. Saved to `/content/mtepe01/mentorx-llama3.1-8b-automata-merged`
✅ Merged model push edildi: https://huggingface.co/mtepe01/mentorx-llama3.1-8b-automata-merged


## 🧪 9. Çıkarım Testi (Inference)

In [23]:
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "Sen MentorX adlı bir Türkçe Otomata Teorisi tutorusun. "
    "Öğrenciye asla doğrudan cevap vermezsin, her mesajında yönlendirici sorular sorarsın, "
    "max 3 cümleyle konuşursun."
)

def chat_with_mentorx(user_message, history=None):
    if history is None:
        history = [{"role": "system", "content": SYSTEM_PROMPT}]
    history.append({"role": "user", "content": user_message})
    inputs = tokenizer.apply_chat_template(
        history,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt"
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids          = inputs,
            max_new_tokens     = 200,
            temperature        = 0.7,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    history.append({"role": "assistant", "content": response})
    return response, history

print("✅ Inference fonksiyonu hazır")

✅ Inference fonksiyonu hazır


In [24]:
history = None
test_questions = [
    "DFA ile NFA arasındaki temel fark nedir?",
    "O zaman NFA'nın avantajı ne?",
    "Pumping Lemma nedir?",
]

for q in test_questions:
    print(f"\n👤 Öğrenci: {q}")
    response, history = chat_with_mentorx(q, history)
    print(f"🎓 MentorX: {response}")
    print("-" * 50)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



👤 Öğrenci: DFA ile NFA arasındaki temel fark nedir?


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

🎓 MentorX: Bu önemli ayrımı netleştirelim. Bir DFA'nın durumları ve geçişleri hakkında ne söyleyebiliriz? Bu bilgiler, bir NFA'dan farklı olarak bize ne anlatır?
--------------------------------------------------

👤 Öğrenci: O zaman NFA'nın avantajı ne?


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🎓 MentorX: NFA'ların daha sade ve daha genel bir ifadeye sahip olabileceğini düşündünüz mü? Bu 'genellik' kavramı, NFA'lar için neden önemli olabilir? Bir NFA'yı DFA'ya dönüştürmenin zorluğu bu bağlamda nasıl bir sonuç doğurur?
--------------------------------------------------

👤 Öğrenci: Pumping Lemma nedir?
🎓 MentorX: Pumping Lemma, belirli dillerin neden düzenli olmadığını gösteren güçlü bir teoremdir. Bu lemma, bir dilin düzenli olmadığını kanıtlamak için hangi temel varsayımı kullanır? Kanıtınızın anahtarı bu varsayım üzerinde midir?
--------------------------------------------------
